# Prepare First-Pass SCEPTER Inputs

This notebook starts the SCEPTER part of the ERW MRV workflow. It assumes the cropland validation notebook has produced cleaned field parcels and uses those parcels as spatial model units.

At this stage we are not yet running SCEPTER. We are preparing clean, auditable input tables: one table for model units, one table for ERW scenarios, and one expanded run table with every model-unit/scenario combination.


## Workflow

1. **Load cleaned cropland parcels** from notebook `03_validate_cropland_outputs.ipynb`.
2. **Create model units** from those parcels, including area and centroid coordinates.
3. **Attach interim soil and climate assumptions** so the table has the fields SCEPTER preparation will need.
4. **Define ERW scenarios** such as baseline, 10 t/ha, 20 t/ha, and 40 t/ha basalt applications.
5. **Expand model runs** so every parcel/model unit is paired with every scenario.
6. **Write staging files** under `data/scepter_runs/inputs/`.

The soil and climate values here are placeholders. Before model results are decision-grade, replace them with SoilGrids, CHIRPS/rainfall, terrain, and local management data.


In [ ]:
from pathlib import Path
import os
import sys

import geopandas as gpd
import pandas as pd

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import SCEPTER_INPUTS, ensure_dir
from erw_mrv.scepter import (
    DEFAULT_SCENARIOS,
    ScepterDefaults,
    expand_scepter_runs,
    load_cleaned_parcels,
    make_model_units,
    model_units_table,
    write_scepter_inputs,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


## Configure Inputs

The preferred input is the cleaned parcel layer from notebook `03`. If you want a quick test before validation is complete, you can point `CLEANED_PARCELS_PATH` to the candidate parcel layer from notebook `02`, but the cleaned layer is the better default.


In [ ]:
OUTPUT_TAG = "jan_jun_2026"
LANDCOVER_DIR = PROJECT_ROOT / "data" / "processed" / "landcover"
VALIDATION_DIR = LANDCOVER_DIR / "validation" / OUTPUT_TAG

CLEANED_PARCELS_PATH = VALIDATION_DIR / f"field_parcels_{OUTPUT_TAG}_cleaned.gpkg"
CANDIDATE_PARCELS_PATH = LANDCOVER_DIR / f"field_parcels_{OUTPUT_TAG}_paper_adapted.gpkg"

if CLEANED_PARCELS_PATH.exists():
    PARCELS_PATH = CLEANED_PARCELS_PATH
    print(f"Using cleaned parcels: {PARCELS_PATH}")
elif CANDIDATE_PARCELS_PATH.exists():
    PARCELS_PATH = CANDIDATE_PARCELS_PATH
    print(f"Using candidate parcels because cleaned parcels are not ready: {PARCELS_PATH}")
else:
    raise FileNotFoundError(
        "No cropland parcel layer found. Run notebook 02 and preferably notebook 03 first. "
        f"Checked: {CLEANED_PARCELS_PATH} and {CANDIDATE_PARCELS_PATH}"
    )

SCEPTER_INPUT_DIR = ensure_dir(SCEPTER_INPUTS)
SCEPTER_INPUT_DIR


## Load Parcels

Each parcel becomes a candidate model unit. For large AOIs, start with a limited number of units while testing the model pipeline, then scale up.


In [ ]:
MAX_TEST_UNITS = 100  # set to None when ready to prepare every cleaned parcel

parcels = load_cleaned_parcels(PARCELS_PATH)
print(f"Loaded parcels: {len(parcels):,}")
print(f"Total parcel area: {parcels['area_ha'].sum():,.2f} ha")
parcels[["area_ha"]].describe()


## Create Model Units

These units include geometry, area, centroid coordinates, and interim soil/climate assumptions. The assumptions are intentionally explicit so they are easy to replace later.


In [ ]:
defaults = ScepterDefaults(
    soil_ph=6.2,
    cec_cmolc_kg=14.0,
    clay_pct=28.0,
    bulk_density_g_cm3=1.30,
    soil_depth_cm=30.0,
    temperature_c=24.0,
    precipitation_mm_yr=1200.0,
    runoff_mm_yr=300.0,
    basalt_application_t_ha=20.0,
    basalt_d50_um=50.0,
    simulation_years=10,
)

model_units = make_model_units(parcels, defaults=defaults, max_units=MAX_TEST_UNITS)
model_unit_table = model_units_table(model_units)

print(f"Model units prepared: {len(model_units):,}")
model_unit_table.head()


## Define Scenarios

The scenarios table controls amendment rate, particle size, and simulation length. The baseline scenario is important because ERW results should be compared against no-amendment conditions.


In [ ]:
scenarios = DEFAULT_SCENARIOS.copy()
scenarios


## Expand To Run Table

This creates one row for every model unit and scenario. Later, each `run_id` can map to a SCEPTER configuration folder or input file.


In [ ]:
runs = expand_scepter_runs(model_unit_table, scenarios)
print(f"SCEPTER runs staged: {len(runs):,}")
runs.head()


## Write SCEPTER Staging Inputs


In [ ]:
written = write_scepter_inputs(
    model_units,
    output_dir=SCEPTER_INPUT_DIR,
    scenarios=scenarios,
)
written


## Quick QA

Check that every scenario has the same number of model units and that no required values are missing.


In [ ]:
qa_by_scenario = runs.groupby("scenario_id").agg(
    run_count=("run_id", "count"),
    total_area_ha=("area_ha", "sum"),
    mean_application_t_ha=("basalt_application_t_ha", "mean"),
).reset_index()

required_columns = [
    "run_id",
    "model_unit_id",
    "scenario_id",
    "area_ha",
    "soil_ph",
    "temperature_c",
    "precipitation_mm_yr",
    "basalt_application_t_ha",
    "basalt_d50_um",
    "simulation_years",
]
missing_values = runs[required_columns].isna().sum().reset_index()
missing_values.columns = ["column", "missing_count"]

qa_by_scenario, missing_values


## Outputs From This Notebook

This notebook writes first-pass SCEPTER input staging files under `data/scepter_runs/inputs/`:

- `scepter_model_units.gpkg`: spatial cropland model units derived from cleaned parcel polygons.
- `scepter_model_units.csv`: non-spatial model unit table with area, centroid, soil, and climate attributes.
- `scepter_scenarios.csv`: baseline and ERW application scenarios.
- `scepter_runs.csv`: one row per model unit and scenario, with a unique `run_id`.
- `README_scepter_inputs.md`: notes explaining the staging files and current assumptions.

The next SCEPTER step is to convert each row in `scepter_runs.csv` into the exact SCEPTER configuration/input-file format and then run a small batch to confirm the model executes cleanly.
